# SoftLUT Shift Test (PYNQ-Z2)

This notebook checks whether writing truth tables to the SoftLUT gate address space causes real inference changes.

## Required files on board
- SoftLUT overlay bit/hwh pair (same basename, for example `actual_llnn.bit` + `actual_llnn.hwh`)
- `softlut_shift_test.py` copied to `/home/xilinx/` or `/home/xilinx/pynq_notebooks/`
- Optional: `weights.bin` or `weights.json` to restore your trained state after testing


In [ ]:
import os
import sys
import importlib.util

# ---- Configure these paths ----
BITSTREAM_PATH = "/home/xilinx/actual_llnn.bit"
WEIGHTS_BIN = None   # e.g. "/home/xilinx/weights.bin"
WEIGHTS_JSON = None  # e.g. "/home/xilinx/weights.json"
IP_NAME = None       # keep None for auto-detect, or set explicit key from overlay.ip_dict

# Put this helper file on the board and point to it directly
HELPER_PATH_CANDIDATES = [
    "/home/xilinx/softlut_shift_test.py",
    "/home/xilinx/pynq_notebooks/softlut_shift_test.py",
    "/home/xilinx/jupyter_notebooks/softlut_shift_test.py",
]

helper_path = next((p for p in HELPER_PATH_CANDIDATES if os.path.exists(p)), None)
if helper_path is None:
    raise FileNotFoundError(
        "Could not find softlut_shift_test.py. Copy it to /home/xilinx, /home/xilinx/pynq_notebooks, or /home/xilinx/jupyter_notebooks."
    )

spec = importlib.util.spec_from_file_location("softlut_shift_test", helper_path)
mod = importlib.util.module_from_spec(spec)
sys.modules[spec.name] = mod  # required for dataclass + postponed annotations
spec.loader.exec_module(mod)
SoftLUTShiftTester = mod.SoftLUTShiftTester

print(f"Loaded helper: {helper_path}")


## 1) Load Overlay + Validate SoftLUT AXI Interface


In [ ]:
tester = SoftLUTShiftTester(BITSTREAM_PATH, ip_name=IP_NAME)
print(f"IP name     : {tester.ip_name}")
print(f"Base address: 0x{tester.base_addr:08X}")
print(f"Total gates : {tester.total_gates}")
print(f"Busy flag   : {tester.is_busy()}")


## 2) Optional: Load Known Weights Before Test

Skip if you only want raw programming checks.

In [ ]:
if WEIGHTS_BIN and os.path.exists(WEIGHTS_BIN):
    tester.load_weights_bin(WEIGHTS_BIN)
    print("Loaded weights.bin")
elif WEIGHTS_JSON and os.path.exists(WEIGHTS_JSON):
    tester.load_weights_json(WEIGHTS_JSON)
    print("Loaded weights.json")
else:
    print("No weights file loaded")


## 3) Scan Gates for Observable Shift Effect

The test compares outputs after programming the same gate with two different truth tables.
If outputs differ for any input vector, the shift path is functionally active.

In [ ]:
NUM_VECTORS = 128
MAX_GATES_TO_SCAN = 256
SEED = 7

vectors = tester.random_input_vectors(NUM_VECTORS, seed=SEED)
gate_stop = min(tester.total_gates, MAX_GATES_TO_SCAN)

results = tester.scan_for_effect(
    vectors,
    gate_start=0,
    gate_stop=gate_stop,
    table_pairs=((0x00000000, 0xFFFFFFFF), (0xAAAAAAAA, 0x55555555)),
    stop_after_first=False,
)

print(f"Scanned gates: [0, {gate_stop})")
print(f"Responsive gates found: {len(results)}")

ranked = sorted(results, key=lambda r: r.num_changed, reverse=True)
for r in ranked[:10]:
    print(
        f"gate={r.gate_id:4d} changed={r.num_changed:3d}/{NUM_VECTORS} "
        f"tables=0x{r.table_lo:08X}/0x{r.table_hi:08X}"
    )


## 4) Inspect One Gate in Detail

In [ ]:
gate_id = ranked[0].gate_id if ranked else 0
detail = tester.gate_effect(gate_id, vectors, 0xAAAAAAAA, 0x55555555)

print(f"gate_id={detail.gate_id}")
print(f"num_changed={detail.num_changed}/{NUM_VECTORS}")
print("first changed vector indices:", detail.changed_indices[:20])

if detail.num_changed:
    idx = detail.changed_indices[0]
    print(f"example change @ vector {idx}: {detail.outputs_lo[idx]} -> {detail.outputs_hi[idx]}")


## 5) Optional: Restore Weights After Testing

Gate programming is destructive. Run this cell if you loaded a trained weights file.

In [ ]:
if WEIGHTS_BIN and os.path.exists(WEIGHTS_BIN):
    tester.load_weights_bin(WEIGHTS_BIN)
    print("Restored weights.bin")
elif WEIGHTS_JSON and os.path.exists(WEIGHTS_JSON):
    tester.load_weights_json(WEIGHTS_JSON)
    print("Restored weights.json")
else:
    print("No weights restore file configured")
